In [ ]:
# %% [markdown]
# # Phase 5: The Distance Geometry of Air Quality
#
# **Technical Spine Project**: How distance metrics change our understanding of similarity.

# %%
import sys
sys.path.append('../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from distance_analyzer import GeometricAnalyzer
from features import get_feature_matrix

print(" Distance Geometry Analysis")

# %%
# Load and prepare data
X_features = pd.read_csv('../data/processed/feature_matrix.csv',
                         index_col='datetime', parse_dates=True)

# Get feature matrix for analysis
X_analysis, feature_names = get_feature_matrix(X_features)
print(f" Data shape: {X_analysis.shape}")
print(f" Features: {len(feature_names)} total")

# Select key features for analysis (avoid too many)
key_features = [
    'CO(GT)', 'PT08.S1(CO)', 'NOx(GT)', 'NO2(GT)',
    'hour_sin', 'hour_cos', 'CO_to_NOx', 'is_weekend'
]

# Use only features that exist
available_features = [f for f in key_features if f in X_analysis.columns]
print(f" Analyzing {len(available_features)} features: {available_features}")

# %%
# Initialize analyzer
analyzer = GeometricAnalyzer(X_analysis, available_features)

# %%
# Compute distance matrices
print("\n1. Computing distance matrices...")
results = analyzer.compute_distance_matrices(
    metrics=['euclidean', 'cosine', 'cityblock', 'correlation'],
    subsample=800  # For reasonable computation time
)

# %%
# Compare nearest neighbors
print("\n2. Comparing nearest neighbors...")
agreement = analyzer.compare_nearest_neighbors(k=10)
print("\nNeighbor Agreement Matrix (k=10):")
print(agreement.round(3))

# %%
# Visualize the complete analysis
print("\n3. Creating comprehensive visualization...")
fig = analyzer.visualize_analysis(
    save_path='../outputs/figures/05_distance_geometry_full.png'
)

# %%
# Generate and display report
print("\n4. Generating technical report...")
report = analyzer.generate_report()

print("\n GEOMETRIC ANALYSIS REPORT")
print("=" * 60)
print(f"Analysis Date: {report['analysis_date']}")
print(f"Samples Analyzed: {report['n_samples']}")
print(f"Features Used: {report['n_features']}")
print(f"Metrics Tested: {', '.join(report['metrics_tested'])}")
print(f"\n RECOMMENDATIONS:")
print(f"  • For CLUSTERING: Use {report['best_clustering_metric']}")
print(f"  • For NEAREST NEIGHBOR: Use {report['best_nn_metric']}")
print(f"\n Key Features Analyzed:")
for i, feat in enumerate(report['feature_names'][:8], 1):
    print(f"  {i}. {feat}")

# %%
# Case study: Show how metrics disagree
print("\n5. CASE STUDY: How metrics change 'similar days'")
print("-" * 50)

# Get a specific day
sample_date = X_analysis.index[100]  # Arbitrary day
sample_idx = 100

# Find neighbors with different metrics
dist_matrices = analyzer.results['distance_matrices']
metrics = analyzer.results['metrics']

print(f"\nAnalyzing: {sample_date}")
print(f"CO(GT) on this day: {X_analysis.iloc[sample_idx]['CO(GT)']:.2f}")

for i, metric in enumerate(metrics):
    distances = dist_matrices[i][sample_idx]
    nearest_idx = np.argsort(distances)[1:6]  # Top 5 neighbors

    print(f"\n{metric.upper()} metric suggests these similar days:")
    for rank, idx in enumerate(nearest_idx, 1):
        neighbor_date = X_analysis.index[idx]
        days_diff = (neighbor_date - sample_date).days
        co_value = X_analysis.iloc[idx]['CO(GT)']
        print(f"  {rank}. {neighbor_date.date()} ({abs(days_diff)} days away, CO: {co_value:.2f})")

# %%
# Save detailed results
print("\n Saving detailed results...")

# Save agreement matrix
agreement.to_csv('../outputs/tables/distance_agreement_matrix.csv')
print("  • Agreement matrix saved")

# Save metric properties
analyzer.results['metric_properties'].to_csv('../outputs/tables/metric_properties.csv', index=False)
print("  • Metric properties saved")

# Generate markdown report
report_text = f"""
# Distance Geometry Analysis Report

## Executive Summary
This analysis compares distance metrics on air quality data to understand how
"similarity" is metric-dependent.

## Key Findings
1. **Best for clustering**: {report['best_clustering_metric']}
2. **Best for nearest neighbor**: {report['best_nn_metric']}
3. **Highest disagreement**: Between {report['metrics_tested'][0]} and {report['metrics_tested'][2]}

## Data Details
- Samples analyzed: {report['n_samples']}
- Features: {report['n_features']}
- Date range: {X_analysis.index.min().date()} to {X_analysis.index.max().date()}

## Recommendations
1. Use different metrics for different tasks
2. Always validate with domain knowledge
3. Consider metric learning for optimal results
"""

with open('../outputs/distance_geometry_report.md', 'w') as f:
    f.write(report_text)

print("  • Markdown report saved")

# %%
# Final insights
print("\n FINAL INSIGHTS: The Geometry of Air Quality")
print("=" * 60)
print("\n1. METRIC CHOICE MATTERS:")
print("   • Euclidean: Sensitive to absolute concentrations")
print("   • Cosine: Finds similar pollution profiles (ratios)")
print("   • Cityblock: Robust to outliers")
print("   • Correlation: Ignores magnitude, focuses on pattern")

print("\n2. PRACTICAL IMPLICATIONS:")
print("   • For regulation: Use Euclidean (absolute levels matter)")
print("   • For source identification: Use Cosine (profile matching)")
print("   • For anomaly detection: Use Correlation (pattern deviation)")

print("\n3. THIS IS YOUR TECHNICAL SPINE:")
print("   • You've moved beyond 'which model is most accurate'")
print("   • You're asking: 'How do we properly measure similarity?'")
print("   • This is what separates practitioners from experts")

print("\n🚀 Next: Apply this same framework to:")
print("   • Energy consumption data")
print("   • Customer churn data")
print("   • Any tabular dataset to build your geometric intuition")

ModuleNotFoundError: No module named 'src'